# Olist Delivery Delay Prediction
### Brazilian E-Commerce Public Dataset | Capstone Project
#### Notebook 02: Data Merging

---

**Authors:** Sura and Aman  
**Verified by:** Ameed and Ruaa

## Goal

Combine all cleaned tables into a single dataset where:

One row = one order

This dataset will be used for:
- Exploratory Data Analysis (EDA)
- Feature Engineering
- Machine Learning modeling

---

### Design Principles

- Preserve row count during joins (anchor = orders)
- Avoid data leakage
- Maintain interpretability
- Separate modeling data from analysis-heavy data (e.g. geolocation)

---

## Load Cleaned Tables

All datasets were cleaned in Notebook 01.  
We reload them here with proper date parsing.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

PATH = '../data/processed/'

orders = pd.read_csv(PATH + 'orders_clean.csv', parse_dates=[
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
])

items = pd.read_csv(PATH + 'items_clean.csv', parse_dates=['shipping_limit_date'])
payments = pd.read_csv(PATH + 'payments_clean.csv')
reviews = pd.read_csv(PATH + 'reviews_clean.csv')
customers = pd.read_csv(PATH + 'customers_clean.csv')
sellers = pd.read_csv(PATH + 'sellers_clean.csv')
products = pd.read_csv(PATH + 'products_clean.csv')
geolocation = pd.read_csv(PATH + 'geolocation_clean.csv')
translation = pd.read_csv(PATH + 'translation_clean.csv')

tables = {
    'orders': orders, 'items': items, 'payments': payments,
    'reviews': reviews, 'customers': customers, 'sellers': sellers,
    'products': products, 'geolocation': geolocation, 'translation': translation
}

for name, df in tables.items():
    print(f'{name:<15} {df.shape[0]:>10,} rows | {df.shape[1]:>2} columns')

orders              96,461 rows | 13 columns
items              112,650 rows |  9 columns
payments           103,886 rows |  5 columns
reviews             99,224 rows |  8 columns
customers           99,441 rows |  6 columns
sellers              3,095 rows |  5 columns
products            32,949 rows |  9 columns
geolocation      1,000,163 rows |  7 columns
translation             71 rows |  2 columns


---

## Geolocation Strategy

The geolocation dataset contains ~1M rows (multiple coordinates per zip code).

### Problem
Direct join → duplicates → breaks "one row per order"

### Solution
Aggregate to one coordinate per zip code using mean (centroid approximation)

### Design Decision
We keep:
- Aggregated version → for modeling
- Full dataset → for spatial analysis in EDA

In [2]:
geo_lookup = (
    geolocation
    .groupby('geolocation_zip_code_prefix', as_index=False)
    .agg(
        lat=('geolocation_lat', 'mean'),
        lng=('geolocation_lng', 'mean')
    )
)

customer_geo = geo_lookup.rename(columns={
    'geolocation_zip_code_prefix': 'customer_zip_code_prefix',
    'lat': 'customer_lat',
    'lng': 'customer_lng'
})

seller_geo = geo_lookup.rename(columns={
    'geolocation_zip_code_prefix': 'seller_zip_code_prefix',
    'lat': 'seller_lat',
    'lng': 'seller_lng'
})

print("Unique zip codes:", len(geo_lookup))

Unique zip codes: 19015


### Result

- Reduced ~1M rows → ~19K zip codes  
- Achieved one-to-one join compatibility  

This preserves dataset structure while retaining geographic signal.

---

## Prepare Items

Items table contains one row per item → must be aggregated to order level.

---

### Fix Missing Translations

We detect missing categories programmatically.

In [3]:
missing_categories = (
    set(products['product_category_name'].dropna()) 
    - set(translation['product_category_name'])
)
missing_categories.discard('unknown')

print("Missing categories:", missing_categories)

Missing categories: {'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}


### Result

Missing categories identified:
- pc_gamer
- portateis_cozinha_e_preparadores_de_alimentos

'unknown' is excluded (represents missing data, not real category).

---

### Apply Translation Patch

In [4]:
missing_cats = {
    'pc_gamer': 'pc_gamer',
    'portateis_cozinha_e_preparadores_de_alimentos': 'portable_kitchen_food_preparators',
}

for pt, en in missing_cats.items():
    if pt not in translation['product_category_name'].values:
        translation = pd.concat([
            translation,
            pd.DataFrame([{
                'product_category_name': pt,
                'product_category_name_english': en
            }])
        ], ignore_index=True)

print("Translation size:", len(translation))

Translation size: 73


### Result

Translation mapping updated successfully.  

All real categories now covered.

---

### Enrich + Aggregate

In [5]:
items_enriched = (
    items
    .merge(products, on='product_id', how='left')
    .merge(translation, on='product_category_name', how='left')
    .merge(sellers[['seller_id','seller_zip_code_prefix','seller_state','seller_city']],
           on='seller_id', how='left')
)

items_enriched['product_volume_cm3'] = (
    items_enriched['product_length_cm'] *
    items_enriched['product_height_cm'] *
    items_enriched['product_width_cm']
)

items_agg = (
    items_enriched
    .groupby('order_id', as_index=False)
    .agg(
        n_items=('order_item_id','max'),
        n_unique_sellers=('seller_id','nunique'),
        total_price=('price','sum'),
        total_freight=('freight_value','sum'),
        avg_product_weight_g=('product_weight_g','mean'),
        avg_product_volume_cm3=('product_volume_cm3','mean'),
        product_category=('product_category_name_english',
                          lambda x: x.mode()[0] if not x.mode().empty else 'unknown'),
        seller_zip_code_prefix=('seller_zip_code_prefix','first'),
        seller_state=('seller_state','first')
    )
)

print("Items → Orders:", len(items_agg))

Items → Orders: 98666


### Result

- Reduced 112,650 rows → ~98K orders  
- Successfully converted to order-level features  

Aggregation preserves key business signals:
- basket size
- seller diversity
- pricing & shipping

---

## Prepare Payments

Multiple rows per order → aggregate to single row

In [6]:
payments_agg = (
    payments
    .groupby('order_id', as_index=False)
    .agg(
        total_payment_value=('payment_value','sum'),
        n_payment_methods=('payment_sequential','max'),
        max_installments=('payment_installments','max'),
        payment_type=('payment_type',
              lambda x: x.mode()[0] if not x.mode().empty else 'unknown')
    )
)

print("Payments → Orders:", len(payments_agg))

Payments → Orders: 99440


### Result

Payments successfully aggregated to order level.  
Captures:
- payment behavior
- complexity (installments)

---

## Prepare Reviews

We keep only `review_score` and ensure one row per order.

In [7]:
reviews_slim = (
    reviews
    .drop_duplicates(subset='order_id', keep='first')
    [['order_id','review_score']]
)

print("Reviews → Orders:", len(reviews_slim))

Reviews → Orders: 98673


### Result

Duplicate reviews removed.  
Using first review as simplification (EDA may refine later).

---

## Build Final Dataset

We merge all tables using LEFT JOIN (anchor = orders).

In [8]:
df = orders.copy()

df = df.merge(customers[['customer_id','customer_unique_id',
                         'customer_zip_code_prefix','customer_state','customer_city']],
              on='customer_id', how='left')

df = df.merge(items_agg, on='order_id', how='left')
df = df.merge(payments_agg, on='order_id', how='left')
df = df.merge(reviews_slim, on='order_id', how='left')
df = df.merge(customer_geo, on='customer_zip_code_prefix', how='left')
df = df.merge(seller_geo, on='seller_zip_code_prefix', how='left')

print("Shape:", df.shape)

Shape: (96461, 35)


### Result

- Row count preserved: 96,461  
- All features successfully merged  
- Dataset ready for validation

---

### Quality Checks

In [9]:
print("Rows:", len(df))
print("Duplicates:", df['order_id'].duplicated().sum())

late_rate = df['is_late'].mean()*100
print("Late rate:", late_rate)

print(df.isnull().sum().sort_values(ascending=False))

Rows: 96461
Duplicates: 0
Late rate: 8.113123438488094
review_score                     646
customer_lng                     264
customer_lat                     264
seller_lng                       215
seller_lat                       215
avg_product_weight_g              16
avg_product_volume_cm3            16
total_payment_value                1
n_payment_methods                  1
max_installments                   1
payment_type                       1
n_unique_sellers                   0
seller_state                       0
seller_zip_code_prefix             0
product_category                   0
total_freight                      0
total_price                        0
order_id                           0
customer_id                        0
is_late                            0
order_status                       0
order_purchase_timestamp           0
order_approved_at                  0
order_delivered_carrier_date       0
order_delivered_customer_date      0
order_estimated_deli

### Result

- No duplicate orders  
- Late delivery rate ≈ 8.1% (consistent with dataset behavior)  
- Missing values identified → handled next

---

### Missing Values

In [10]:
df['has_review'] = df['review_score'].notna().astype(int)

df['product_category'] = df['product_category'].fillna('unknown')

df['avg_product_weight_g'] = df['avg_product_weight_g'].fillna(df['avg_product_weight_g'].median())
df['avg_product_volume_cm3'] = df['avg_product_volume_cm3'].fillna(df['avg_product_volume_cm3'].median())

df['n_payment_methods'] = df['n_payment_methods'].fillna(0).astype(int)
df['max_installments'] = df['max_installments'].fillna(0).astype(int)
df['total_payment_value'] = df['total_payment_value'].fillna(0)
df['payment_type'] = df['payment_type'].fillna('unknown')

### Result

- Missing values handled appropriately  
- Median used for robustness (reduces outlier impact)  
- Review absence converted into feature (`has_review`)

---

### Remove Canceled

In [11]:
# df[df['order_status']=='canceled'][['order_delivered_customer_date']]

keep this for EDA Team 

---

### Save

In [12]:
df.to_csv('../data/processed/olist_merged.csv', index=False)

print("Saved:", df.shape)
print("Late %:", df['is_late'].mean()*100)

Saved: (96461, 36)
Late %: 8.113123438488094


### Final Result

- Final dataset: 96,461 rows  
- One row per order  
- Clean, consistent, ready for EDA and modeling

---

## Final Note

Geolocation data (~1M rows) was aggregated to one coordinate per zip code (mean)
to preserve the one-row-per-order structure.

This avoids data duplication and ensures a valid dataset for modeling.
Using raw geolocation would break the dataset via one-to-many joins.

This approach is standard practice in Olist analyses and similar datasets.

The full geolocation data is preserved separately for spatial EDA.

---

The final dataset integrates all key features (orders, customers, items, payments, reviews)
and is designed to support both:

- reliable machine learning modeling  
- flexible, multi-dimensional EDA

The objective is to produce a clean, consistent, and analysis-ready dataset.

---

In [21]:
# Preview the dataset
import pandas as pd

# Load final merged dataset
df = pd.read_csv('../data/processed/olist_merged.csv', parse_dates=[
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
])

#  Shape and columns
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


Shape: (96461, 36)

Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'is_late', 'is_delivered', 'delivery_status', 'has_timestamp_issue', 'delivery_days', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_state', 'customer_city', 'n_items', 'n_unique_sellers', 'total_price', 'total_freight', 'avg_product_weight_g', 'avg_product_volume_cm3', 'product_category', 'seller_zip_code_prefix', 'seller_state', 'total_payment_value', 'n_payment_methods', 'max_installments', 'payment_type', 'review_score', 'customer_lat', 'customer_lng', 'seller_lat', 'seller_lng', 'has_review']


In [22]:
# Basic info
print("\nInfo:")
print(df.info())


Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96461 entries, 0 to 96460
Data columns (total 36 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       96461 non-null  object        
 1   customer_id                    96461 non-null  object        
 2   order_status                   96461 non-null  object        
 3   order_purchase_timestamp       96461 non-null  datetime64[ns]
 4   order_approved_at              96461 non-null  datetime64[ns]
 5   order_delivered_carrier_date   96461 non-null  datetime64[ns]
 6   order_delivered_customer_date  96461 non-null  datetime64[ns]
 7   order_estimated_delivery_date  96461 non-null  datetime64[ns]
 8   is_late                        96461 non-null  int64         
 9   is_delivered                   96461 non-null  int64         
 10  delivery_status                96461 non-null  object        
 11  has_time

In [23]:
#  Quick descriptive stats for numeric columns
print("\nNumeric Summary:")
print(df.describe().T)


Numeric Summary:
                                 count                           mean  \
order_purchase_timestamp         96461  2018-01-01 23:53:26.642249216   
order_approved_at                96461  2018-01-02 10:10:06.480142336   
order_delivered_carrier_date     96461  2018-01-05 05:21:04.508827392   
order_delivered_customer_date    96461  2018-01-14 13:17:13.228102400   
order_estimated_delivery_date    96461  2018-01-25 17:33:14.236012544   
is_late                        96461.0                       0.081131   
is_delivered                   96461.0                       0.999938   
has_timestamp_issue            96461.0                       0.014234   
delivery_days                  96461.0                      12.093582   
customer_zip_code_prefix       96461.0                   35198.925825   
n_items                        96461.0                       1.142223   
n_unique_sellers               96461.0                       1.013902   
total_price                    96

In [24]:
#  Check missing values
print("\nMissing values per column:")
print(df.isnull().sum().sort_values(ascending=False))



Missing values per column:
review_score                     646
customer_lng                     264
customer_lat                     264
seller_lng                       215
seller_lat                       215
order_id                           0
seller_zip_code_prefix             0
total_freight                      0
avg_product_weight_g               0
avg_product_volume_cm3             0
product_category                   0
n_payment_methods                  0
seller_state                       0
total_payment_value                0
customer_id                        0
max_installments                   0
payment_type                       0
total_price                        0
n_unique_sellers                   0
n_items                            0
is_late                            0
order_status                       0
order_purchase_timestamp           0
order_approved_at                  0
order_delivered_carrier_date       0
order_delivered_customer_date      0
order_esti

### Handle Missing Values 

In [25]:
# Numeric columns
df['customer_lat'] = df['customer_lat'].fillna(df['customer_lat'].median())
df['customer_lng'] = df['customer_lng'].fillna(df['customer_lng'].median())
df['seller_lat'] = df['seller_lat'].fillna(df['seller_lat'].median())
df['seller_lng'] = df['seller_lng'].fillna(df['seller_lng'].median())
df['avg_product_weight_g'] = df['avg_product_weight_g'].fillna(df['avg_product_weight_g'].median())
df['avg_product_volume_cm3'] = df['avg_product_volume_cm3'].fillna(df['avg_product_volume_cm3'].median())
df['total_payment_value'] = df['total_payment_value'].fillna(0)
df['n_payment_methods'] = df['n_payment_methods'].fillna(0)
df['max_installments'] = df['max_installments'].fillna(0)

# Categorical
df['payment_type'] = df['payment_type'].fillna('unknown')

---

In [26]:
# Preview first 5 rows
print("\nFirst 5 rows:")
print(df.head())


First 5 rows:
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp   order_approved_at  \
0    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37 2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49 2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06 2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39 2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00         

---

In [27]:
print("Rows:", df.shape[0])
print("Unique orders:", df['order_id'].nunique())

Rows: 96461
Unique orders: 96461


In [28]:
df.isna().mean().sort_values(ascending=False).head(15)

review_score              0.006697
order_id                  0.000000
total_payment_value       0.000000
total_freight             0.000000
avg_product_weight_g      0.000000
avg_product_volume_cm3    0.000000
product_category          0.000000
seller_zip_code_prefix    0.000000
seller_state              0.000000
n_payment_methods         0.000000
customer_id               0.000000
max_installments          0.000000
payment_type              0.000000
customer_lat              0.000000
customer_lng              0.000000
dtype: float64

## Final Dataset
Each row represents one order with aggregated features
from items, payments, reviews, and geolocation.